# PART 03 · 5장 작업형 2 제출 파일 생성 — 연습 문제 정답

교재 298쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)
from sklearn.metrics import f1_score, mean_absolute_error

## 문제 01

multiclass_ticket_priority 데이터로 priority_level을 예측하시오. ticket_id는 입력 변수에서 제외하고, 검증 데이터의 Macro F1을 확인한 뒤 전체 train으로 다시 학습하여 test의 클래스값을 result.csv의 pred 컬럼으로 저장하시오.

**정답 코드**

In [2]:
multi_base = 'data/part03_type2/multiclass_ticket_priority'
multi_train = pd.read_csv(multi_base + '/train.csv')
multi_test = pd.read_csv(multi_base + '/test.csv')
multi_submit = pd.read_csv(multi_base + '/sample_submission.csv')

target = 'priority_level'
X_raw = multi_train.drop(columns=[target, 'ticket_id']).copy()
y = multi_train[target].copy()
X_test_raw = multi_test.drop(columns=['ticket_id']).copy()

# 1단계 검증: train을 나눠 Macro F1을 확인한다.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=0, stratify=y
)

num_cols = X_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_train_raw[col].median()
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_train_raw[col].mode()[0]
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)

X_train = pd.get_dummies(X_train_raw)
X_val = pd.get_dummies(X_val_raw).reindex(
    columns=X_train.columns, fill_value=0
)

model = RandomForestClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)
val_pred = model.predict(X_val)
macro_f1 = f1_score(y_val, val_pred, average='macro')
print('Macro F1:', round(macro_f1, 4))

# 2단계 제출: 전체 train으로 다시 전처리하고 다시 학습한다.
X_full = X_raw.copy()
X_test_full = X_test_raw.copy()

num_cols = X_full.select_dtypes(exclude='object').columns
cat_cols = X_full.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_full[col].median()
    X_full[col] = X_full[col].fillna(fill_value)
    X_test_full[col] = X_test_full[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_full[col].mode()[0]
    X_full[col] = X_full[col].fillna(fill_value)
    X_test_full[col] = X_test_full[col].fillna(fill_value)

X_full = pd.get_dummies(X_full)
X_test_full = pd.get_dummies(X_test_full).reindex(
    columns=X_full.columns, fill_value=0
)

final_model = RandomForestClassifier(n_estimators=100, random_state=0)
final_model.fit(X_full, y)

result = multi_submit.copy()
result['pred'] = final_model.predict(X_test_full)
result.to_csv('result.csv', index=False)

check = pd.read_csv('result.csv')
print('shape:', check.shape)
print('columns:', check.columns.tolist())
print(check['pred'].value_counts().sort_index())

Macro F1: 0.4595


shape: (260, 1)
columns: ['pred']
pred
0    103
1    152
2      5
Name: count, dtype: int64


**간단 해설**

검증 단계에서는 X_train에서 결측치 대체 기준을 계산하고 X_val을 X_train의 더미 컬럼에 맞춘다. 최종 제출 단계에서는 전체 train 기준으로 전처리를 다시 수행한다. Macro F1 문제이므로 test에는 확률이 아니라 클래스값을 저장한다.

## 문제 02

regression_delivery_minutes 데이터에서 delivery_minutes가 NaN이거나 0인 행을 학습 대상에서 제외한 뒤 배달 시간을 예측하시오. order_id는 입력 변수에서 제외하고, 검증 데이터의 MAE를 확인한 다음 전체 유효 train으로 다시 학습하여 test의 연속형 예측값을 result.csv의 pred 컬럼으로 저장하시오.

**정답 코드**

In [3]:
reg_base = 'data/part03_type2/regression_delivery_minutes'
reg_train = pd.read_csv(reg_base + '/train.csv')
reg_test = pd.read_csv(reg_base + '/test.csv')
reg_submit = pd.read_csv(reg_base + '/sample_submission.csv')

target = 'delivery_minutes'
valid_cond = (reg_train[target].notna() & (reg_train[target] != 0))
train_valid = reg_train.loc[valid_cond].copy()
print('before:', reg_train.shape[0], 'after:', train_valid.shape[0])

X_raw = train_valid.drop(columns=[target, 'order_id']).copy()
y = train_valid[target].copy()
X_test_raw = reg_test.drop(columns=['order_id']).copy()

# 1단계 검증: MAE를 확인한다.
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=0
)

num_cols = X_train_raw.select_dtypes(exclude='object').columns
cat_cols = X_train_raw.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_train_raw[col].median()
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_train_raw[col].mode()[0]
    X_train_raw[col] = X_train_raw[col].fillna(fill_value)
    X_val_raw[col] = X_val_raw[col].fillna(fill_value)

X_train = pd.get_dummies(X_train_raw)
X_val = pd.get_dummies(X_val_raw).reindex(
    columns=X_train.columns, fill_value=0
)

model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(X_train, y_train)
val_pred = model.predict(X_val)
print('MAE:', round(mean_absolute_error(y_val, val_pred), 4))

# 2단계 제출: 전체 유효 train으로 다시 학습한다.
X_full = X_raw.copy()
X_test_full = X_test_raw.copy()

num_cols = X_full.select_dtypes(exclude='object').columns
cat_cols = X_full.select_dtypes(include='object').columns
for col in num_cols:
    fill_value = X_full[col].median()
    X_full[col] = X_full[col].fillna(fill_value)
    X_test_full[col] = X_test_full[col].fillna(fill_value)
for col in cat_cols:
    fill_value = X_full[col].mode()[0]
    X_full[col] = X_full[col].fillna(fill_value)
    X_test_full[col] = X_test_full[col].fillna(fill_value)

X_full = pd.get_dummies(X_full)
X_test_full = pd.get_dummies(X_test_full).reindex(
    columns=X_full.columns, fill_value=0
)

final_model = RandomForestRegressor(n_estimators=100, random_state=0)
final_model.fit(X_full, y)

result = reg_submit.copy()
result['pred'] = final_model.predict(X_test_full)
result.to_csv('result.csv', index=False)

check = pd.read_csv('result.csv')
print('shape:', check.shape)
print('columns:', check.columns.tolist())
print(check['pred'].head())

before: 840 after: 840


MAE: 7.0549


shape: (280, 1)
columns: ['pred']
0    58.515
1    59.835
2    51.318
3    42.108
4    45.147
Name: pred, dtype: float64


**간단 해설**

유효한 target 행을 train_test_split() 전에 확정하고, 검증 단계와 최종 제출 단계에서 같은 train_valid를 사용한다. 회귀 문제이므로 pred에는 연속형 예측값이 들어간다.

---

**정답** 1. Macro F1은 0.4595이며, result.csv는 260행 1열이고 컬럼명은 pred다, 2. MAE는 7.0549이며 result.csv는 280행 1열이다.